# Segundo Entregable: dashboard y modelos base

**Predicción de brotes de dengue en Colombia con cuatro semanas de anticipación**

**Maestría en Matemáticas · Proyecto de Aprendizaje Automático · Daniel Serrano Romero**

Este entregable tiene dos partes:

1. **Un dashboard en Dash + Plotly** con tres pestañas: **Contexto del problema**, **EDA** y **Modelos base** (regresión logística y
   SVM lineal). Cada resultado del dashboard trae su propia interpretación.
2. **Este Jupyter Book**, el informe del desarrollo: un capítulo por pestaña con las mismas figuras del dashboard (importadas del mismo
   código), su interpretación y los **detalles que no aparecen en el dashboard**: fórmulas de las pruebas, tablas completas, código de
   los Pipelines, el escalado de Platt de la SVM, la comparación de las funciones de pérdida y la arquitectura de la aplicación.

## Resumen de resultados

| Elemento | Resultado |
|---|---|
| Datos | 376 municipios × 782 semanas (2010–2024); temperatura imputada en el 64 % de las semanas (MAR estructural) |
| Objetivo | Brote en $t+4$ según el canal endémico; ≈ 10 % de brotes en entrenamiento, del 1,7 % al 25 % según el año |
| EDA | La historia de casos domina; el clima actúa con rezago y de forma no lineal; El Niño cuadruplica el riesgo; Moran ≈ 0,31 y Knox significativo |
| Modelos base | Regresión logística: AUC-ROC 0,885 y AUC-PR 0,809 · SVM lineal: 0,887 y 0,808 (prueba 2023–2024) |
| Líneas base | Persistencia de 8 semanas 0,859 / 0,728 · persistencia 0,787 / 0,588 · Dummy 0,500 / 0,303 |
| Diagnóstico | Sin sobreajuste; residuos con autocorrelación temporal y espacial; probabilidades subestimadas por el cambio de prevalencia |

## Cómo leer este libro

| Capítulo | Pestaña del dashboard |
|---|---|
| 1. Contexto del problema | Contexto del problema |
| 2. EDA I: datos, objetivo y relaciones | EDA › Datos y calidad, Variable objetivo, Univariado, Bivariado, Multivariado, Fuga de datos |
| 3. EDA II: tiempo, espacio y preprocesamiento | EDA › Temporal, Espacial, Espacio-temporal, Preprocesamiento |
| 4. Modelos base | Modelos base (todas las sub-pestañas) |
| 5. El dashboard | Arquitectura, componentes interactivos y ejecución |
| 6. Conclusiones | Hallazgos, limitaciones y reproducibilidad |

El análisis completo paso a paso (todas las pruebas con su código) está en el libro del **Primer Entregable**, del que este
entregable reutiliza el código (`entregable1/e1`).


# Contexto del problema

Este capítulo acompaña la pestaña **Contexto del problema** del dashboard: interpreta cada elemento y agrega detalles que no caben en el dashboard (la construcción del objetivo, la comparación de las funciones de pérdida de los dos modelos base y la verificación de la partición).

In [1]:
import os  # Rutas
import sys  # Configuración del intérprete
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable2 (paquete tablero)
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath("../../entregable1"))  # Paquete e1 (código fuente que se muestra)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
from IPython.display import HTML, Markdown, display  # Incrustar figuras y texto
from tablero import figuras as fg  # Las mismas figuras del dashboard
from tablero.datos import A, M, MOD, FEATURES, curvas, metricas_umbral, rezago_espacial  # Los mismos artefactos del dashboard
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales


def mostrar(fig):  # Muestra una figura del dashboard como HTML interactivo
    """Pone el fondo oscuro de las tarjetas del dashboard y carga Plotly desde internet."""
    fig.update_layout(paper_bgcolor=fg.SUPERFICIE, plot_bgcolor=fg.SUPERFICIE, margin=dict(l=40, r=20, t=50, b=40))  # Fondo oscuro
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False, config={"displaylogo": False})))  # Figura interactiva
k = A["contexto"]["kpi"]  # Indicadores de la portada
pd.Series({"Casos 2010–2024": f"{k['casos']:,}", "Municipios": k["municipios"], "Departamentos": k["departamentos"], "Semanas": k["semanas"],
           "Año récord": f"{k['anio_record']} ({k['casos_record']:,} casos)", "Prevalencia de brote (entrenamiento)": f"{k['prev_tr']:.1%}",
           "Prevalencia de brote (validación 2022)": f"{k['prev_va']:.1%}", "Predictoras": k["p"]}, name="valor").to_frame()

,valor
Casos 2010–2024,"1,261,746"
Municipios,376
Departamentos,32
Semanas,782
Año récord,"2024 (261,382 casos)"
Prevalencia de brote (entrenamiento),10.3%
Prevalencia de brote (validación 2022),12.4%
Predictoras,34


**Interpretación de los indicadores.** El panel reúne 1,26 millones de casos notificados en 376 municipios durante 782 semanas. 2024 es el año récord y queda dentro de la prueba, así que los modelos se evalúan en un régimen más extremo que el de entrenamiento. La clase brote es minoritaria (≈ 10 %), lo que orienta la elección de métricas.

## Pregunta y objetivos

**Pregunta.** ¿Es posible anticipar, con cuatro semanas de margen, si un municipio entrará en brote de dengue usando solo la información disponible en la semana actual?

El dashboard presenta cuatro objetivos: describir la base de datos y su estructura; definir el brote sin información del futuro; explorar la relación de la historia de casos, el clima y El Niño con los brotes; y entrenar dos modelos base lineales comparados con líneas base triviales.

## Casos por año

In [2]:
mostrar(fg.fig_anual(A["contexto"]["anual"]))  # Misma figura de la pestaña Contexto
A["contexto"]["anual"].to_frame("casos").T

ANO,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
casos,129914,25373,45608,108991,87256,81334,84571,21222,38407,104393,66615,43054,55335,108291,261382


**Interpretación.** Las epidemias de 2010, 2013, 2019 y 2023–2024 dominan la serie y coinciden con episodios de El Niño. 2017 y 2018 son años de muy baja transmisión. Esta alternancia es la razón de que la prevalencia del objetivo cambie tanto entre años.

## Detalle: construcción del objetivo con el canal endémico

El dashboard muestra la fórmula; aquí está el código que la implementa (`src/datos.py`). El umbral se calcula de forma vectorizada: para cada semana se toman los 25 valores históricos (5 años × 5 semanas) y su percentil 75.

In [3]:
import inspect  # Código fuente
from src import datos as sd  # Módulo del canal endémico
print(inspect.getsource(sd.canal_endemico))  # Función que calcula el umbral

def canal_endemico(casos, semanas_por_mun):  # Umbral superior del canal endémico (método de cuartiles, Bortman 1999)
    """Q3 de los casos en las mismas semanas (± VENTANA) de los ANIOS_HISTORIA años anteriores, por municipio."""
    n_mun = len(casos) // semanas_por_mun  # Número de municipios
    C = casos.reshape(n_mun, semanas_por_mun)  # Matriz municipios × semanas
    desfases = np.array([-52 * k + d for k in range(1, cf.ANIOS_HISTORIA + 1)  # Misma semana de cada año previo…
                         for d in range(-cf.VENTANA_SEMANAS, cf.VENTANA_SEMANAS + 1)])  # …y sus vecinas (± 2)
    umbral = np.full(C.shape, np.nan)  # Sin historia suficiente queda nulo
    inicio = 52 * cf.ANIOS_HISTORIA + cf.VENTANA_SEMANAS  # Primera semana con 5 años completos de historia
    columnas = np.arange(inicio, semanas_por_mun)  # Semanas con umbral calculable
    indices = columnas[:, None] + desfases[None, :]  # Para cada semana, las 25 semanas históricas de referencia
    umbral[:, column

In [4]:
serie = A["objetivo"]["canal"][76001]  # Cali
mostrar(fg.fig_canal(serie, "Santiago de Cali"))  # Canal endémico
print(f"Semanas de brote en Cali 2015–2021: {int(serie['brote'].sum())} de {len(serie)} ({serie['brote'].mean():.1%})")  # Resumen

Semanas de brote en Cali 2015–2021: 98 de 366 (26.9%)


**Interpretación.** El umbral (línea punteada) sigue la historia de cada municipio: en Cali es alto porque la transmisión es habitual. Un brote no es "muchos casos", sino **más casos de lo normal para esa época del año en ese municipio**. El requisito de al menos 3 casos evita que un solo caso en un municipio sin historia cuente como brote.

## Detalle: las funciones de pérdida de los dos modelos base

La regresión logística minimiza la pérdida logarítmica; la SVM lineal (`LinearSVC`) minimiza la bisagra al cuadrado. Ambas son funciones convexas del margen $z = \tilde y\,(\mathbf w^\top \mathbf x + b)$ con $\tilde y \in \{-1, 1\}$:

$$\ell_{\text{log}}(z) = \ln(1 + e^{-z}), \qquad \ell_{\text{bisagra}^2}(z) = \max(0,\, 1 - z)^2.$$

La diferencia clave: la bisagra es **exactamente cero** para los puntos bien clasificados con margen ($z \ge 1$), así que la SVM solo aprende de los puntos cercanos a la frontera (vectores de soporte); la logística sigue aprendiendo de todos los puntos y produce probabilidades directamente. Por eso la SVM necesita un paso adicional (escalado de Platt) para dar probabilidades.

In [5]:
import plotly.graph_objects as go  # Figura
z = np.linspace(-3, 3, 300)  # Márgenes
fig = go.Figure()  # Comparación de pérdidas
fig.add_scatter(x=z, y=np.log1p(np.exp(-z)), name="Logística: ln(1 + e^(−z))", line=dict(color=fg.AZUL, width=3))  # Logística
fig.add_scatter(x=z, y=np.maximum(0, 1 - z) ** 2, name="Bisagra al cuadrado: max(0, 1 − z)²", line=dict(color=fg.MAGENTA, width=3))  # SVM
fig.add_scatter(x=z, y=(z < 0).astype(float), name="Pérdida 0-1", line=dict(color=fg.GRIS, dash="dot"))  # Referencia
fig.update_layout(xaxis_title="margen z = ỹ (wᵀx + b)", yaxis_title="pérdida", yaxis_range=[0, 5])  # Ejes
mostrar(fg.tema(fig, 380))  # Muestra

**Interpretación.** Las dos pérdidas son sustitutos convexos de la pérdida 0-1 (que no se puede optimizar directamente). Para márgenes negativos (errores) ambas crecen; para márgenes grandes la logística decae suavemente y la bisagra se anula. Con datos tabulares, muchas filas y las mismas variables, ambas suelen encontrar hiperplanos muy parecidos, como se verá en el capítulo de modelos base.

## Detalle: la partición se verifica, no se supone

In [6]:
mostrar(fg.fig_particion(A["contexto"]["linea_particion"]))  # Línea de tiempo
A["contexto"]["particion"]  # Filas y fechas de cada periodo

,filas,desde,hasta
periodo,,,
historia,96256,2010-01-04,2014-11-24
entrenamiento,136112,2014-12-29,2021-11-29
validación,18048,2022-01-03,2022-11-28
prueba,37600,2023-01-02,2024-11-25
purgada,6016,2014-12-01,2024-12-23


**Interpretación.** Las 6 016 filas *purgadas* son las cuatro semanas de cada frontera, cuya etiqueta cae en el periodo siguiente (por ejemplo, diciembre de 2021, cuyo objetivo es enero de 2022). Con la purga, la última etiqueta del entrenamiento es anterior a la primera semana de la validación, y lo mismo entre validación y prueba: ninguna etiqueta se usa fuera de su periodo.

## Detalle: consideraciones éticas

El panel contiene solo conteos agregados; no requiere consentimiento ni comité de ética. Los cruces finos (edad × sexo × semana en municipios pequeños) no se publican para evitar cualquier riesgo de reidentificación. El riesgo más relevante es de **equidad**: si la notificación es peor donde hay menos recursos, el modelo aprenderá menos riesgo justo donde la vigilancia es más débil.

# EDA I: datos, objetivo y relaciones entre variables

Acompaña las sub-pestañas **Datos y calidad, Variable objetivo, Univariado, Bivariado, Multivariado y Fuga de datos** del dashboard. Cada figura es la misma del dashboard; después de cada una va su interpretación y, en los recuadros *Detalle*, lo que el dashboard no muestra: las fórmulas de las pruebas y las tablas completas. Todas las decisiones usan las semanas 2010–2021.

In [7]:
import os  # Rutas
import sys  # Configuración del intérprete
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable2 (paquete tablero)
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath("../../entregable1"))  # Paquete e1 (código fuente que se muestra)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
from IPython.display import HTML, Markdown, display  # Incrustar figuras y texto
from tablero import figuras as fg  # Las mismas figuras del dashboard
from tablero.datos import A, M, MOD, FEATURES, curvas, metricas_umbral, rezago_espacial  # Los mismos artefactos del dashboard
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales


def mostrar(fig):  # Muestra una figura del dashboard como HTML interactivo
    """Pone el fondo oscuro de las tarjetas del dashboard y carga Plotly desde internet."""
    fig.update_layout(paper_bgcolor=fg.SUPERFICIE, plot_bgcolor=fg.SUPERFICIE, margin=dict(l=40, r=20, t=50, b=40))  # Fondo oscuro
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False, config={"displaylogo": False})))  # Figura interactiva
cal = A["calidad"]  # Calidad de los datos
print(f"Temperatura imputada: {cal['frac_imputada']:.1%} de las semanas")  # Faltante oculto
cal["patron"].to_frame("municipios")  # Patrón por municipio

Temperatura imputada: 64.3% de las semanas


,municipios
temp_mean_missing,
siempre imputada,226
parcialmente,129
nunca imputada,21


## Datos y calidad

### Patrón de los faltantes

In [8]:
mostrar(fg.fig_matriz_faltantes(cal["matriz_faltantes"]))  # Matriz municipio × año
mostrar(fg.fig_temp_imputada(cal["hist_temp"]))  # Observada frente a imputada

**Interpretación.** No hay celdas nulas, pero la temperatura está imputada en el 64 % de las semanas. El patrón es por **municipio** (226 nunca medidos, 21 siempre medidos, 129 parcialmente) y estable en el tiempo. La imputación original es casi una constante cerca de 22,4 °C: no conserva la diferencia climática entre municipios.

### Mecanismo de los faltantes

:::{admonition} Detalle: prueba de Little
Con $J$ patrones de faltantes, $\boldsymbol\mu$ y $\boldsymbol\Sigma$ estimadas por máxima verosimilitud (algoritmo EM) y $\bar{\mathbf x}_j$ la media de las variables observadas en el patrón $j$:

$$d^2 = \sum_{j=1}^{J} n_j\,(\bar{\mathbf x}_j - \boldsymbol\mu_j)^\top \boldsymbol\Sigma_j^{-1} (\bar{\mathbf x}_j - \boldsymbol\mu_j) \sim \chi^2\Big(\sum_j p_j - p\Big)\ \text{bajo MCAR}.$$

La implementación (EM incluido) está en `entregable1/e1/estadistica.py`.
:::

In [9]:
li = cal["little"]  # Prueba de Little
print(f"Little: d² = {li['d2']:,.1f}, gl = {li['gl']}, p = {li['p']:.1e}")  # Resultado
pd.Series(cal["smd"], name="diferencia estandarizada (faltante − observado)").to_frame()  # Tamaños de efecto

Little: d² = 2,304.8, gl = 5, p = 0.0e+00


,diferencia estandarizada (faltante − observado)
prec_total,-0.0149
ndvi_mean,0.2808
oni_anom,0.0080
log_poblacion,-0.6801
log_casos,-0.3347


**Interpretación.** Little rechaza MCAR. La diferencia más grande es el tamaño del municipio ($d \approx -0{,}7$): faltan las mediciones en municipios pequeños, sin estación meteorológica. Es **MAR estructural**; un componente MNAR no es verificable. Decisión: conservar la bandera `temp_missing` (método del indicador) en lugar de reimputar con vecinos.

### Valores imposibles, atípicos y representatividad

In [10]:
pd.Series(cal["chequeos"], name="filas con problema").to_frame()  # Consistencia

,filas con problema
Filas duplicadas exactas,0
"Claves (municipio, semana) repetidas",0
Suma de edades ≠ total,0
Hombres + mujeres ≠ total,0
"Temperatura fuera de [0, 40] °C",0
"NDVI fuera de [−1, 1]",0
Conteos o lluvia negativos,0
Semanas que no avanzan 7 días,0


In [11]:
display(cal["atipicos"].to_frame())  # Atípicos por la regla de Tukey
cal["top_casos"]  # Las semanas con más casos

,atípicos (%)
casos_totales,11.7997
temp_mean,18.1375
prec_total,4.6682
ndvi_mean,1.6621
oni_anom,2.3018
Flujo_in,12.5646
poblacion,14.0957


,municipio,departamento,week_start,casos_totales
0,Santiago De Cali,Valle Del Cauca,2024-05-20,1426
1,Santiago De Cali,Valle Del Cauca,2024-06-03,1399
2,Santiago De Cali,Valle Del Cauca,2024-05-27,1378
3,Santiago De Cali,Valle Del Cauca,2024-06-17,1366
4,Santiago De Cali,Valle Del Cauca,2024-06-10,1364
5,Santiago De Cali,Valle Del Cauca,2024-04-01,1284
6,Santiago De Cali,Valle Del Cauca,2024-04-29,1281
7,Santiago De Cali,Valle Del Cauca,2024-04-15,1277


In [12]:
rep = cal["representatividad"]  # Cobertura por región
print(f"Población cubierta en 2024: {cal['pob_cubierta'] / 1e6:.1f} millones")  # Cobertura poblacional
rep

Población cubierta en 2024: 32.2 millones


,país,panel,cobertura,participación panel,participación país
region,,,,,
Andina,629,173,0.2750,0.4601,0.5606
Caribe,197,99,0.5025,0.2633,0.1756
Pacífica,179,44,0.2458,0.1170,0.1595
Orinoquía,59,41,0.6949,0.1090,0.0526
Amazonía,58,19,0.3276,0.0505,0.0517


**Interpretación.** No hay valores imposibles ni duplicados. Los atípicos de casos son epidemias reales (Cali en 2024) y se conservan con transformación logarítmica; el 18 % de "atípicos" de temperatura es un artefacto de la imputación. El panel cubre un tercio de los municipios pero ≈ 60 % de la población, con el Caribe y la Orinoquía sobrerrepresentados.

## Variable objetivo

In [13]:
o = A["objetivo"]  # Objetivo
mostrar(fg.fig_prevalencia_anual(o["anual"]))  # Prevalencia por año
mostrar(fg.fig_mes_region(o["mes"], o["region"]))  # Por mes y región
o["anual"].rename(columns={"mean": "prevalencia", "sum": "brotes", "size": "filas"})

,prevalencia,brotes,filas
ANO,,,
2015,0.1281,2119,16544
2016,0.0906,1772,19552
2017,0.0167,326,19552
2018,0.0724,1416,19552
2019,0.2532,4951,19552
2020,0.0992,1977,19928
2021,0.0600,1083,18048


**Interpretación.** La prevalencia del brote va del 1,7 % (2017) al 25 % (2019): el desbalance depende del año. La exactitud no sirve para decidir (no alertar nunca acierta el 90 %); se usan AUC-PR, MCC y Brier, y la validación recorre años completos. La estacionalidad mensual del objetivo es débil porque el canal endémico ya la descuenta.

In [14]:
mostrar(fg.fig_objetivo_tiempo(o["nacional"]))  # Municipios en brote
mapa = M.merge(o["mapa"].reset_index(), on="COD_MUN_N")  # Prevalencia por municipio
mostrar(fg.fig_mapa_valor(mapa, "prevalencia", "brote", ".0%", (0, 0.3)))  # Mapa
mapa["prevalencia"].describe().to_frame().T

,count,mean,std,min,25%,50%,75%,max
prevalencia,376.0000,0.1028,0.0848,0.0000,0.0340,0.0822,0.1671,0.4108


**Interpretación.** Los brotes llegan en ondas nacionales (2015–2016 y 2019). En el espacio la prevalencia varía entre municipios (mediana ≈ 8 %), pero no sigue el nivel de casos, porque cada municipio se compara consigo mismo.

## Análisis unidimensional

In [15]:
u = A["univariado"]  # Univariado
for v in ["casos_totales", "temp_mean", "prec_total", "ndvi_mean"]:  # Variables representativas
    mostrar(fg.fig_histograma(u["hist"][v], v))  # Histograma + caja
u["tabla"]  # Estadísticos, forma, atípicos y normalidad

,media,mediana,desv. est.,p5,p95,asimetría,curtosis,atípicos IQR (%),p normalidad (K²)
variable,,,,,,,,,
casos_totales,3.5549,0.0000,20.2744,0.0000,12.0000,18.2714,473.3343,10.0979,0.0000
temp_mean,23.1829,22.4633,3.0148,18.2039,28.6500,-0.2163,2.1400,18.1569,0.0000
prec_total,42.5322,25.1000,51.8553,0.0000,145.0000,2.2587,8.3742,4.6377,0.0000
ndvi_mean,0.6307,0.6502,0.1147,0.4119,0.7852,-0.7837,0.4180,1.6132,0.0000
oni_anom,0.0238,-0.1400,0.8715,-1.2000,1.9100,0.8874,1.1323,4.3131,0.0000
oni_total,27.0263,27.1300,0.9927,25.4300,28.8300,0.1406,-0.4647,0.0000,0.0000
Flujo_in,2.9785,0.5896,16.4615,0.0841,9.5308,20.5873,576.3910,12.5374,0.0000
poblacion,"85,757.9309","22,490.0000","410,379.4074","6,233.0000","299,173.0000",14.3056,233.9893,14.0957,0.0000


**Interpretación.** Los casos tienen asimetría > 18 y curtosis en cientos (mediana 0): se usa $\log(1+x)$. La temperatura es bimodal por la imputación; la lluvia es asimétrica a la derecha; el NDVI, levemente a la izquierda. Ninguna variable es normal (todas las pruebas rechazan aun con 5 000 filas), lo que justifica pruebas no paramétricas.

In [16]:
pd.DataFrame({"departamentos raros (< 3 municipios)": [", ".join(u["categoricas"]["departamento"][u["categoricas"]["departamento"] < 3].index)],
              "semanas por fase ENSO": [u["categoricas"]["enso"].to_dict()], "municipios por región": [u["categoricas"]["region"].to_dict()]}).T

,0
departamentos raros (< 3 municipios),"Guainía, Quindío, Vaupés, Bogotá, D.C., Guavia..."
semanas por fase ENSO,"{'Neutral': 287, 'La Niña': 179, 'El Niño': 160}"
municipios por región,"{'Amazonía': 19, 'Andina': 173, 'Caribe': 99, ..."


**Interpretación.** El departamento tiene categorías raras y no se codifica (haría memorizar lugares); la semana 53 (2015 y 2020) se resuelve con seno y coseno; ninguna fase ENSO es rara.

## Análisis bidimensional

:::{admonition} Detalle: medidas de asociación usadas
* **Correlación biserial de rangos** (Mann-Whitney): $r_{rb} = 2U/(n_1 n_0) - 1 \in [-1, 1]$; es $2\,\text{AUC} - 1$.
* **Kruskal-Wallis** con tamaño de efecto $\varepsilon^2 = H/(n-1)$ (0,01 pequeño, 0,08 mediano, 0,26 grande).
* **V de Cramér** con corrección de sesgo: $V = \sqrt{\tilde\varphi^2/\min(\tilde k - 1, \tilde r - 1)}$.
* **Información mutua** $I(X;Y) = \sum p(x,y)\ln\frac{p(x,y)}{p(x)p(y)}$, estimada por vecinos más cercanos; capta relaciones no lineales.
* **VIF** $= 1/(1-R_j^2) = [\mathbf R^{-1}]_{jj}$, con $\mathbf R$ la matriz de correlación.
* **Holm** controla la tasa de error de la familia; **Benjamini-Hochberg**, la tasa de falsos descubrimientos.
:::

In [17]:
b = A["bivariado"]  # Bivariado
for p in b["pares"]:  # Pares numéricos
    x, y = p.split("|")  # Variables
    mostrar(fg.fig_densidad(b["pares"][p], x, y))  # Densidad conjunta
pd.DataFrame({p: {"Pearson": d["pearson"], "Spearman": d["spearman"], "n": d["n"]} for p, d in b["pares"].items()}).T

,Pearson,Spearman,n
temp_mean_m4|prec_total_m4,0.0555,-0.0942,"48,070.0000"
lc_m8|lc_l0,0.9105,0.7803,"132,728.0000"
temp_mean_m8|lc_m8,0.0509,0.0353,"48,070.0000"
oni_l8|lc_m8,0.1576,0.1884,"132,728.0000"


**Interpretación.** Temperatura y lluvia están poco relacionadas; la historia de casos es muy persistente (Pearson 0,91 frente a Spearman 0,78: la correlación lineal la dominan las semanas epidémicas); la temperatura casi no se correlaciona con los casos; el ONI rezagado sí, de forma positiva.

In [18]:
mostrar(fg.fig_correlacion(b["corr"]))  # Spearman
b["vif"].sort_values(ascending=False).to_frame("VIF").T  # Multicolinealidad

,temp_mean_m8,temp_mean_m12,temp_mean_m4,lc_m8,temp_mean_m1,lc_m12,ndvi_mean_m8,ndvi_mean_m4,prec_total_m8,ndvi_mean_m12,prec_total_m12,ndvi_mean_m1,prec_total_m4,lc_l3,lc_l2,...,flujo_m4,lpob,lc_l0,brote_l0,prec_total_m1,l_umbral_obj,incid,prec_anom_m8,ndvi_anom_m8,lc_m4,temp_missing,temp_anom_m8,ratio_umbral,sem_sin,sem_cos
VIF,"1,026.9446",525.3422,357.4618,122.0889,79.4624,61.5032,55.6818,29.8368,27.3910,24.2848,15.2052,9.5105,8.7865,6.5461,6.3211,...,4.5551,4.2437,4.2049,3.4583,2.4502,2.2941,1.9784,1.5736,1.5219,1.4684,1.3404,1.2660,1.2402,1.1576,1.1439


**Interpretación.** Las medias móviles de una misma variable forman bloques con $|\rho| > 0{,}9$ y VIF ≫ 10 (hasta ≈ 1 000 en la temperatura de 8 semanas). La predicción no sufre, pero los coeficientes individuales son inestables: se usa penalización L2.

In [19]:
mostrar(fg.fig_asociacion(b["asociacion"], "Información mutua"))  # Asociación
b["asociacion"].sort_values("mi", ascending=False)  # Tabla completa (no aparece en el dashboard)

,r_rb,p,auc,mi,p_holm,q_fdr
variable,,,,,,
brote_m8,0.7323,0.0000,0.8662,0.1192,0.0000,0.0000
incid,0.6932,0.0000,0.8466,0.1076,0.0000,0.0000
ratio_umbral,0.6123,0.0000,0.8061,0.1072,0.0000,0.0000
lc_m4,0.7625,0.0000,0.8812,0.1040,0.0000,0.0000
lc_m8,0.7539,0.0000,0.8769,0.1023,0.0000,0.0000
lc_m12,0.7376,0.0000,0.8688,0.0933,0.0000,0.0000
lc_l0,0.7044,0.0000,0.8522,0.0924,0.0000,0.0000
brote_l0,0.5402,0.0000,0.7701,0.0911,0.0000,0.0000
lc_l1,0.6828,0.0000,0.8414,0.0880,0.0000,0.0000


**Interpretación.** La historia de casos domina (AUC univariados de 0,77–0,88). El flujo, el ONI y la población tienen asociación moderada. El clima local tiene efectos pequeños aunque casi todo es significativo tras Holm y FDR: con $n$ grande el tamaño de efecto manda.

In [20]:
for v in ["temp_mean_m8", "prec_total_m8", "oni_l8", "lc_m4"]:  # Relaciones no lineales
    mostrar(fg.fig_deciles(b["deciles"][v], v))  # Tasa de brote por decil

**Interpretación.** Con la temperatura medida, el riesgo sube hacia los 25–28 °C y se estabiliza; con la historia de casos crece de forma monótona y fuerte. Las formas no lineales anticipan una limitación de los modelos base lineales.

In [21]:
display(b["kruskal"])  # Por región
display(b["cramer"])  # Categóricas
b["enso"]  # Brote por fase ENSO

,H,p,ε²,p (Holm)
variable,,,,
lc_l0,835.9909,0.0000,0.0063,0.0000
temp_mean_m8,"19,859.6053",0.0000,0.4131,0.0000
prec_total_m8,"15,390.8373",0.0000,0.1160,0.0000
ndvi_mean_m8,"6,353.1595",0.0000,0.0479,0.0000


,categorías,χ²,gl,p,V de Cramér,p (Holm)
variable,,,,,,
MES,12,85.3226,11,0.0000,0.0237,0.0000
region,5,152.5075,4,0.0000,0.0334,0.0000
departamento,32,876.4571,31,0.0000,0.0798,0.0000
fase_enso,3,"3,700.7494",2,0.0000,0.1669,0.0000
temp_missing,2,359.0001,1,0.0000,0.0519,0.0000


y,0,1
fase_enso,,
La Niña,0.9548,0.0452
Neutral,0.9169,0.0831
El Niño,0.8318,0.1682


**Interpretación.** La temperatura difiere muchísimo entre regiones ($\varepsilon^2 \approx 0{,}41$) y la historia de casos casi nada. La fase ENSO es la categórica más asociada al brote (V ≈ 0,17): 16,8 % en El Niño frente a 4,5 % en La Niña.

## Análisis multivariado

In [22]:
mu = A["multivariado"]  # Multivariado
mostrar(fg.fig_pca(mu["varianza"], mu["pca_xy"], mu["pca_y"]))  # PCA
acum = np.cumsum(mu["varianza"])  # Varianza acumulada
print(f"Componentes para 80 %: {np.searchsorted(acum, 0.8) + 1} · para 90 %: {np.searchsorted(acum, 0.9) + 1} · autovalor > 1: {(mu['autovalores'] > 1).sum()}")
mu["cargas"].round(3)  # Cargas completas de las 4 primeras componentes

Componentes para 80 %: 8 · para 90 %: 12 · autovalor > 1: 7


,CP1,CP2,CP3,CP4
lc_l0,0.3060,-0.0260,0.0690,-0.0160
lc_l1,0.3020,-0.0230,0.0620,-0.0080
lc_l2,0.3020,-0.0220,0.0620,-0.0070
lc_l3,0.3010,-0.0210,0.0600,-0.0060
lc_m4,0.3230,-0.0240,0.0670,-0.0100
lc_m8,0.3210,-0.0210,0.0620,-0.0060
lc_m12,0.3170,-0.0180,0.0570,-0.0030
brote_l0,0.2020,-0.0330,0.0780,-0.0630
brote_m8,0.2370,-0.0320,0.0780,-0.0620
ratio_umbral,0.0700,-0.0040,0.0540,-0.0950


**Interpretación.** 8 componentes explican el 80 % de la varianza: la dimensionalidad efectiva es mucho menor que 34. CP1 es la historia de casos; CP2–CP3, temperatura y NDVI; CP4, lluvia. Los brotes no se separan linealmente en dos dimensiones.

In [23]:
mostrar(fg.fig_tsne(mu["tsne"], mu["tsne_y"], mu["tsne_region"], "region"))  # t-SNE por región
display(mu["atipicos"])  # Atípicos multivariados
mu["atipicos_municipios"].to_frame("filas atípicas")

,atípicos (%),tasa de brote en atípicos,tasa de brote en el resto
"Mahalanobis (χ² 99,9 %)",6.3107,0.3147,0.0885
Isolation Forest,14.7135,0.3886,0.0535
Ambos métodos,4.7013,0.3885,0.0887


,filas atípicas
municipio,
"Bogotá, D.C.",306
Santiago De Cali,287
Guamal,223
Florencia,205
Medellín,187
Buenaventura,186
Valledupar,143
Villavicencio,136


**Interpretación.** t-SNE agrupa más por región que por objetivo: el brote es un estado transitorio, no una subpoblación. Los atípicos multivariados tienen una tasa de brote cuatro veces mayor (semanas epidémicas, grandes ciudades): se conservan.

:::{admonition} Detalle: distancia de Mahalanobis
$D^2(\mathbf x) = (\mathbf x - \boldsymbol\mu)^\top \boldsymbol\Sigma^{-1}(\mathbf x - \boldsymbol\mu) \sim \chi^2_{34}$ bajo normalidad; se marca atípico por encima del percentil 99,9. Por la colinealidad se usa la pseudo-inversa de $\boldsymbol\Sigma$.
:::

In [24]:
mostrar(fg.fig_grupos(mu["perfil"]))  # Grupos de municipios
print("Silueta por k:", {k: round(v, 3) for k, v in mu["silueta"].items()})  # Elección de k
mu["perfil"].groupby("grupo")[["prevalencia", "temp", "lluvia", "lpob", "flujo"]].mean()  # Perfil de cada grupo

Silueta por k: {2: 0.327, 3: 0.221, 4: 0.225, 5: 0.226, 6: 0.197, 7: 0.196, 8: 0.188}


,prevalencia,temp,lluvia,lpob,flujo
grupo,,,,,
0,0.0816,23.3201,43.1404,9.7559,0.4192
1,0.1924,22.9541,35.8226,12.0193,1.7437


**Interpretación.** La mejor silueta es 0,33 con $k = 2$: estructura débil a moderada. Los grupos se separan por tamaño y movilidad, no por clima; los municipios grandes y conectados tienen más del doble de semanas en brote.

## Fuga de datos

In [25]:
f = A["fuga"]  # Auditoría
mostrar(fg.fig_auc_univariado(f["auc_univariado"]))  # AUC univariado
f["demo"].to_frame("AUC-ROC de validación cruzada")  # Demostración

,AUC-ROC de validación cruzada
Correcto: años hacia adelante con purga,0.8985
Fuga temporal: semanas mezcladas al azar,0.9064
Fuga directa: casos de la semana objetivo,0.9941


**Interpretación.** Ninguna predictora supera 0,9 de AUC univariado; los casos de la semana objetivo (información del futuro) llegan a ≈ 0,96 y se descartan. Con la validación correcta el AUC es ≈ 0,90; mezclando semanas sube un poco, y con la fuga directa llega a ≈ 0,99. El umbral objetivo $u_{t+4}$ usa semanas $t-258$ a $t-46$, todas anteriores a $t$: no es fuga.

# EDA II: tiempo, espacio y preprocesamiento

Acompaña las sub-pestañas **Temporal, Espacial, Espacio-temporal y Preprocesamiento** del dashboard.

In [26]:
import os  # Rutas
import sys  # Configuración del intérprete
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable2 (paquete tablero)
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath("../../entregable1"))  # Paquete e1 (código fuente que se muestra)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
from IPython.display import HTML, Markdown, display  # Incrustar figuras y texto
from tablero import figuras as fg  # Las mismas figuras del dashboard
from tablero.datos import A, M, MOD, FEATURES, curvas, metricas_umbral, rezago_espacial  # Los mismos artefactos del dashboard
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales


def mostrar(fig):  # Muestra una figura del dashboard como HTML interactivo
    """Pone el fondo oscuro de las tarjetas del dashboard y carga Plotly desde internet."""
    fig.update_layout(paper_bgcolor=fg.SUPERFICIE, plot_bgcolor=fg.SUPERFICIE, margin=dict(l=40, r=20, t=50, b=40))  # Fondo oscuro
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False, config={"displaylogo": False})))  # Figura interactiva
t = A["temporal"]  # Componente temporal
pd.Series({"fuerza de la tendencia": t["fuerza"]["tendencia"], "fuerza estacional": t["fuerza"]["estacional"],
           "municipios con ADF significativo": t["mun_adf"], "municipios con KPSS significativo": t["mun_kpss"]}).to_frame("valor")

,valor
fuerza de la tendencia,0.6610
fuerza estacional,0.0214
municipios con ADF significativo,0.6223
municipios con KPSS significativo,0.4867


## Componente temporal

In [27]:
mostrar(fg.fig_serie(t["nacional"], t["mensual"], A["contexto"]["anual"].loc[:2021], "semana"))  # Serie semanal
mostrar(fg.fig_estacionalidad(t["calendario"]))  # Estacionalidad
mostrar(fg.fig_stl(t["stl"]))  # STL

**Interpretación.** La serie la dominan ondas de varios años (fuerza de la tendencia 0,66) y la estacionalidad anual es casi nula (0,02): la información útil está en el nivel reciente de la serie.

:::{admonition} Detalle: fuerza de la tendencia y de la estacionalidad (Wang, Smith y Hyndman, 2006)
Con la descomposición STL $y_t = T_t + S_t + R_t$: $F_T = \max\big(0, 1 - \frac{\text{Var}(R_t)}{\text{Var}(T_t + R_t)}\big)$ y $F_S = \max\big(0, 1 - \frac{\text{Var}(R_t)}{\text{Var}(S_t + R_t)}\big)$.
:::

In [28]:
t["estacionariedad"]  # ADF (H0: raíz unitaria) y KPSS (H0: estacionaria)

,ADF estadístico,ADF p,KPSS estadístico,KPSS p
serie,,,,
Nivel: log casos,-2.5110,0.1128,0.2516,0.1000
Primera diferencia,-14.9251,0.0000,0.0506,0.1000


**Interpretación.** En nivel, ADF no rechaza la raíz unitaria y KPSS tampoco rechaza la estacionariedad: una serie muy persistente en la que ambas pruebas pierden potencia. La primera diferencia es estacionaria. Por municipio, el 62 % rechaza la raíz unitaria y el 49 % rechaza la estacionariedad.

In [29]:
mostrar(fg.fig_acf(t["acf"], t["pacf"], t["n"]))  # ACF y PACF
mostrar(fg.fig_ccf(t["ccf"]))  # Correlación cruzada
t["ccf"].abs().agg(["idxmax", "max"]).T.rename(columns={"idxmax": "rezago de máxima |r|", "max": "máxima |r|"})

,rezago de máxima |r|,máxima |r|
temp_anom,0.0000,0.0564
prec_anom,15.0000,0.0362
ndvi_anom,12.0000,0.0325
oni_anom,5.0000,0.1717


**Interpretación.** ACF persistente y PACF concentrada en los primeros rezagos: se usan rezagos de 1–3 semanas y medias de 4, 8 y 12. El clima local llega con rezago (lluvia y NDVI 12–15 semanas) y correlaciones pequeñas; el ONI es la señal más fuerte.

In [30]:
mostrar(fg.fig_regimenes(t["stl"], t["cortes"], t["nino"]))  # Regímenes
print("Puntos de cambio:", ", ".join(str(c.date()) for c in t["cortes"]))  # Fechas
display(t["deriva"])  # PSI
mostrar(fg.fig_deriva(t["concepto"]))  # Concept drift
t["concepto"]

Puntos de cambio: 2011-01-03, 2012-12-31, 2016-09-05, 2018-05-28, 2019-05-27, 2020-06-08


,PSI 2015–18 → 2019–21,PSI entrenamiento → 2022
lc_m4,0.0401,0.0068
brote_m8,0.0586,0.0033
ratio_umbral,0.0781,0.0328
temp_mean_m4,0.0029,0.0307
temp_anom_m8,0.0270,0.2010
prec_total_m4,0.0023,0.1049
ndvi_mean_m4,0.0016,0.0199
oni_anom,1.9548,7.4466
oni_l8,1.8518,7.5873
flujo_m4,0.0708,0.0639


,lc_m4,brote_m8,ratio_umbral,oni_l8,temp_anom_m8
2015,0.8307,0.8197,0.7170,0.5961,0.5531
2016,0.8640,0.8643,0.8110,0.6784,0.6437
2017,0.8304,0.7659,0.6764,0.5023,0.5017
2018,0.8862,0.8476,0.7895,0.6528,0.5509
2019,0.8831,0.8617,0.7897,0.4629,0.5085
2020,0.8708,0.8679,0.8263,0.6677,0.5574
2021,0.8915,0.8551,0.7821,0.5744,0.5171
2022,0.8937,0.8673,0.7861,0.5036,0.4587


**Interpretación.** Los regímenes siguen las epidemias y la caída de 2020 (pandemia). La deriva es baja para la historia de casos, moderada hacia 2022 en la temperatura anómala y la lluvia, y muy fuerte en el ONI; el concepto casos → brote es estable (AUC 0,83–0,89 cada año), el del clima no.

:::{admonition} Detalle: índice de estabilidad poblacional y segmentación binaria
$\text{PSI} = \sum_i (a_i - r_i)\ln(a_i/r_i)$ sobre los deciles de la referencia (< 0,1 estable; > 0,25 deriva fuerte). Los puntos de cambio se buscan dividiendo recursivamente la serie donde más baja la suma de cuadrados, con segmentos de al menos 52 semanas y una penalización tipo BIC $3 \cdot 2\hat\sigma^2 \ln n$, con $\hat\sigma^2$ la varianza del residuo de STL.
:::

In [31]:
mostrar(fg.fig_calor(t["calor"], t["calor_regiones"]))  # Panel municipio × semana

**Interpretación.** Ondas sincronizadas (franjas verticales) y focos locales: heterogeneidad entre municipios en nivel y en tiempo.

## Componente espacial

In [32]:
s = A["espacial"]  # Componente espacial
mapa = s["mapa"]  # Municipios
mostrar(fg.fig_cobertura(mapa, s["fuera"]))  # Cobertura
mostrar(fg.fig_mapa_valor(mapa, "log_incidencia", "log incidencia"))  # Incidencia
pd.Series(s["clark_evans"]).to_frame("Clark-Evans")

,Clark-Evans
R,0.7718
r_obs,23.2031
r_esp,30.0644
z,-8.4660
p,0.0000
area,"1,359,415.7535"


**Interpretación.** Las coordenadas DIVIPOLA (EPSG:4326) son válidas. El panel deja vacíos en la Orinoquía y la Amazonía (sesgo de muestreo espacial) y los municipios están agrupados ($R \approx 0{,}77$).

In [33]:
mostrar(fg.fig_ripley(s["ripley"]))  # Ripley
mostrar(fg.fig_dbscan(mapa))  # DBSCAN
mapa["dbscan"].value_counts().head(8).to_frame("municipios")

,municipios
dbscan,
ruido,193
C3,37
C6,32
C7,23
C11,18
C0,13
C1,10
C5,10


:::{admonition} Detalle: índice de Clark-Evans y función K de Ripley
$R = \bar r_{\text{obs}} / \bar r_{\text{esp}}$ con $\bar r_{\text{esp}} = 0{,}5/\sqrt{n/A}$ (aleatoriedad completa); $K(d) = \frac{A}{n(n-1)}\sum_{i \ne j}\mathbb 1(d_{ij} \le d)$ y $L(d) = \sqrt{K(d)/\pi}$. La envolvente se obtiene con 99 simulaciones de puntos uniformes en la misma envolvente convexa, así que el efecto de borde afecta igual a lo observado y a lo simulado.
:::

In [34]:
display(s["sensibilidad"])  # Moran con distintas matrices de pesos
mostrar(fg.fig_moran(mapa, rezago_espacial(mapa["log_incidencia"])))  # Diagrama de Moran
pd.DataFrame(s["moran"]).T

,sin vecinos,I,z,p
matriz,,,,
k = 4,0,0.3319,9.7818,0.0020
k = 6,0,0.3081,11.0876,0.0020
k = 8,0,0.2754,11.7778,0.0020
k = 12,0,0.2546,14.0761,0.0020
banda 50 km,27,0.3274,9.0605,0.0020
banda 100 km,6,0.2060,10.7724,0.0020
banda 200 km,3,0.1430,13.6026,0.0020


,I,E_I,z,p
log_incidencia,0.3081,-0.0027,11.4422,0.0010
prevalencia,0.0453,-0.0027,1.7035,0.0550


**Interpretación.** La incidencia tiene autocorrelación espacial clara (I ≈ 0,31 con k = 6) y robusta a la matriz de pesos. La proporción de semanas en brote no (I ≈ 0,05, p ≈ 0,06): el canal endémico normaliza el nivel de cada zona.

:::{admonition} Detalle: I de Moran, LISA y Getis-Ord
$I = \frac{n}{S_0}\frac{\mathbf z^\top \mathbf W \mathbf z}{\mathbf z^\top \mathbf z}$ con $E[I] = -1/(n-1)$; significancia por 999 permutaciones. LISA: $I_i = \frac{z_i}{m_2}\sum_j w_{ij} z_j$ con permutación condicional. Getis-Ord: $G_i^* = \frac{\sum_j w_{ij}x_j - \bar x \sum_j w_{ij}}{s\sqrt{(n\sum_j w_{ij}^2 - (\sum_j w_{ij})^2)/(n-1)}}$, que ya es un puntaje z.
:::

In [35]:
mostrar(fg.fig_lisa(mapa))  # LISA
mostrar(fg.fig_gi(mapa))  # Gi*
print(f"LISA significativos tras FDR: {s['fdr_lisa']}")  # Corrección
mapa.loc[mapa["LISA"] == "Alto-Alto", "departamento"].value_counts().to_frame("municipios Alto-Alto")

LISA significativos tras FDR: 53


,municipios Alto-Alto
departamento,
Meta,23
Tolima,22
Huila,12
Putumayo,6
Cundinamarca,5
Santander,4
Casanare,3
Boyacá,1


**Interpretación.** 76 municipios Alto-Alto (Meta, Tolima, Huila, Putumayo) y 53 significativos tras FDR: los focos no son un artefacto de las comparaciones múltiples.

In [36]:
mostrar(fg.fig_vario(s["vario"], float(mapa["log_incidencia"].var())))  # Semivariograma
display(s["heterogeneidad"])  # Por región
pd.Series({"MAUP: correlación temperatura–incidencia (municipios)": s["maup"]["municipios"], "MAUP: (departamentos)": s["maup"]["departamentos"],
           "error mediano de la distancia euclidiana sobre grados": s["error_grados"][0]}).to_frame("valor")

,municipios,prevalencia,AUC lc_m4,AUC oni_l8,AUC temp_anom_m8,AUC prec_total_m8
region,,,,,,
Amazonía,19,0.1421,0.8601,0.6257,0.5164,0.4734
Andina,173,0.0961,0.8760,0.6931,0.5732,0.4685
Caribe,99,0.1023,0.8988,0.6153,0.5279,0.5250
Orinoquía,41,0.1083,0.8588,0.6600,0.5278,0.4869
Pacífica,44,0.1084,0.8912,0.6571,0.6327,0.4634


,valor
MAUP: correlación temperatura–incidencia (municipios),0.1039
MAUP: (departamentos),0.2230
error mediano de la distancia euclidiana sobre grados,0.0020


**Interpretación.** Alcance ≈ 100 km con efecto pepita alto. La relación del clima con el brote cambia entre regiones, y al agregar a departamentos la correlación con la temperatura se duplica (MAUP): se trabaja a escala municipal con haversine.

## Componente espacio-temporal

In [37]:
st = A["espacio_temporal"]  # Espacio-temporal
for a in (2017, 2019):  # Un año sin brotes y uno epidémico
    mostrar(fg.fig_anual_mapa(st["anual"], a))  # Mapa del año
mostrar(fg.fig_focos(st["focos"]))  # Focos
st["focos"]["categoría"].value_counts().to_frame("municipios")

,municipios
categoría,
Nunca foco,193
Esporádico,159
Decreciente,8
Emergente / intensificándose,8
Persistente,8


**Interpretación.** El patrón espacial cambia cada año y los focos se desplazan: 159 municipios son focos esporádicos y solo 8 persistentes.

:::{admonition} Detalle: clasificación de focos
Para cada municipio se calcula $G_i^*$ de la log incidencia de cada año (2015–2021) y la prueba de Mann-Kendall sobre esa serie: **persistente** si es foco ≥ 80 % de los años; **emergente** si es foco el último año con tendencia creciente ($p < 0{,}1$); **decreciente** si fue foco con tendencia decreciente; **esporádico** si fue foco sin tendencia.
:::

In [38]:
mostrar(fg.fig_regiones(st["regiones"]))  # Series por región
display(st["corr_regiones"].round(2))  # Sincronía
st["knox"]  # Prueba de Knox

region,Amazonía,Andina,Caribe,Orinoquía,Pacífica
region,,,,,
Amazonía,1.0000,0.7200,0.5900,0.6000,0.3800
Andina,0.7200,1.0000,0.6400,0.6800,0.4700
Caribe,0.5900,0.6400,1.0000,0.6000,0.0100
Orinoquía,0.6000,0.6800,0.6000,1.0000,0.0300
Pacífica,0.3800,0.4700,0.0100,0.0300,1.0000


,distancia (km),tiempo (semanas),observado,esperado,razon,p,pares_espacio
0,25,2,2278,"2,017.4565",1.1291,0.0010,112907
1,50,4,11127,"8,721.3323",1.2758,0.0010,272238
2,100,4,29314,"24,232.8859",1.2097,0.0010,756176
3,100,8,54888,"45,297.0430",1.2117,0.0010,756176
4,200,8,161579,"144,888.3784",1.1152,0.0010,2419132


**Interpretación.** Las regiones están sincronizadas salvo el Pacífico. La prueba de Knox encuentra en todas las escalas más inicios de brote cercanos en espacio y tiempo de lo esperado (razón 1,1–1,3; p = 0,001).

:::{admonition} Detalle: prueba de Knox
Se cuentan los pares de inicios de brote a menos de $d_s$ km y $d_t$ semanas, y se compara con 999 permutaciones de los tiempos entre lugares (que rompen la interacción y conservan los patrones puramente espaciales y temporales).
:::

In [39]:
mostrar(fg.fig_moran_semanal(st["moran_semanal"]))  # Moran semanal
mostrar(fg.fig_cruzada(st["cruzada"]))  # Vecindario → municipio
st["cruzada"]

,vecinos,propio
k,,
0,0.4343,1.0000
1,0.4242,0.6885
2,0.4189,0.6616
3,0.4123,0.6430
4,0.4055,0.6176
5,0.3952,0.5944
6,0.3863,0.5707
7,0.3753,0.5484
8,0.3637,0.5258


**Interpretación.** La autocorrelación espacial de los casos es positiva casi todas las semanas y el vecindario en $t$ anticipa al municipio en $t+4$ (r ≈ 0,41), aunque menos que su propia historia (r ≈ 0,62). Consecuencia: variables de vecindad rezagadas para los siguientes modelos, y validación por zonas nuevas con zona de separación.

## Preprocesamiento

| Paso | Decisión | Hallazgo que la motiva |
|---|---|---|
| Partición | Cronológica con purga de 4 semanas | Autocorrelación; objetivo en $t+4$ |
| Validación | Ventana creciente por años con purga | Prevalencia y régimen cambian por año |
| Imputación | `temp_missing` + `SimpleImputer(mediana)` | Faltantes MAR estructurales |
| Transformación | $\log(1+x)$ de casos, flujo y población | Asimetría > 14 |
| Escalamiento | `StandardScaler` en el Pipeline | Penalización L2 y escalas distintas |
| Codificación | Seno y coseno de la semana; sin departamento | Estacionalidad cíclica; categorías raras |
| Atípicos | Se conservan | Son epidemias reales |
| Rezagos y ventanas | $t \dots t-3$; medias 4, 8 y 12 | ACF y PACF |
| Anomalías | Respecto a la normal mensual del municipio (2010–2021) | Clima muy distinto entre regiones |
| Regularización | L2 con $C$ por validación cruzada | VIF ≫ 10 |
| Desbalance | `class_weight` en la malla; sin SMOTE | ≈ 10 % de brotes, variable |
| Coordenadas | No se usan como predictoras | Focos que se desplazan |

In [40]:
import inspect  # Código fuente
from e1 import modelo_base as mb  # Modelo base del Primer Entregable
print(inspect.getsource(mb.pipeline_base))  # Pipeline de la regresión logística
print(inspect.getsource(mb.ValidacionAnualPurgada))  # Validación cruzada purgada

def pipeline_base():  # Pipeline del modelo base: imputación + estandarización + regresión logística
    return Pipeline([("imputer", SimpleImputer(strategy="median")),  # Mediana del entrenamiento (protege ante nulos)
                     ("scaler", StandardScaler()),  # Coeficientes comparables y regularización justa
                     ("clf", LogisticRegression(max_iter=5000))])  # Regresión logística con penalización L2

class ValidacionAnualPurgada:  # Validación cruzada hacia adelante por años con purga de etiquetas
    """groups = 10 000 · año de la semana t + año de la semana objetivo (se codifican juntos para GridSearchCV)."""

    def __init__(self, anios=(2018, 2019, 2020, 2021)):  # Años de validación
        self.anios = anios  # Guarda los años

    def split(self, X, y=None, groups=None):  # Índices de cada pliegue
        g = np.asarray(groups)  # Códigos
        anio_t, anio_obj = g // 10000, g % 10000  # Decodifica los dos años
        for a in self.anios:  # Un pli

**Interpretación.** Todo el preprocesamiento ajustable (mediana e imputación, media y desviación) vive dentro del Pipeline, así que en cada pliegue se aprende solo con la parte de entrenamiento. La validación cruzada purgada descarta las filas cuyo objetivo cruza la frontera del año validado.

# Modelos base: regresión logística y SVM lineal

Acompaña la pestaña **Modelos base** del dashboard. Los dos modelos comparten el preprocesamiento (`SimpleImputer` → `StandardScaler`), la malla de hiperparámetros ($C$ y `class_weight`), la validación cruzada hacia adelante por años con purga y el criterio de selección (AUC-PR). Se comparan con cuatro líneas base triviales y se evalúan una sola vez en 2023–2024.

In [41]:
import os  # Rutas
import sys  # Configuración del intérprete
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable2 (paquete tablero)
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath("../../entregable1"))  # Paquete e1 (código fuente que se muestra)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
from IPython.display import HTML, Markdown, display  # Incrustar figuras y texto
from tablero import figuras as fg  # Las mismas figuras del dashboard
from tablero.datos import A, M, MOD, FEATURES, curvas, metricas_umbral, rezago_espacial  # Los mismos artefactos del dashboard
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales


def mostrar(fig):  # Muestra una figura del dashboard como HTML interactivo
    """Pone el fondo oscuro de las tarjetas del dashboard y carga Plotly desde internet."""
    fig.update_layout(paper_bgcolor=fg.SUPERFICIE, plot_bgcolor=fg.SUPERFICIE, margin=dict(l=40, r=20, t=50, b=40))  # Fondo oscuro
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False, config={"displaylogo": False})))  # Figura interactiva
import inspect  # Código fuente
pd.DataFrame({n: {"mejores hiperparámetros": MOD["mejores"][n], "tiempo de GridSearchCV (s)": round(MOD["tiempos"][n], 1),
                  "umbral (máximo F1 en 2022)": round(MOD["umbrales"][n], 3)} for n in ["Regresión logística", "SVM lineal"]}).T

,mejores hiperparámetros,tiempo de GridSearchCV (s),umbral (máximo F1 en 2022)
Regresión logística,"{'clf__C': 0.01, 'clf__class_weight': None}",16.1000,0.2440
SVM lineal,"{'clf__C': 0.01, 'clf__class_weight': None}",14.9000,0.2940


## Detalle: el código de los dos Pipelines

La regresión logística viene del Primer Entregable (`e1/modelo_base.py`); la SVM lineal y su escalado de Platt están en `precalculo.py`. `LinearSVC(dual=False)` resuelve el problema primal, más eficiente cuando hay muchas más filas que variables.

In [42]:
import precalculo  # Script de precálculo (SVM)
from e1 import modelo_base as mb  # Modelo base logístico
print(inspect.getsource(mb.pipeline_base))  # Pipeline logístico
print(inspect.getsource(precalculo.pipeline_svm))  # Pipeline SVM

def pipeline_base():  # Pipeline del modelo base: imputación + estandarización + regresión logística
    return Pipeline([("imputer", SimpleImputer(strategy="median")),  # Mediana del entrenamiento (protege ante nulos)
                     ("scaler", StandardScaler()),  # Coeficientes comparables y regularización justa
                     ("clf", LogisticRegression(max_iter=5000))])  # Regresión logística con penalización L2

def pipeline_svm():  # Pipeline de la máquina de vectores de soporte lineal
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),  # Mismo preprocesamiento
                     ("clf", LinearSVC(dual=False, max_iter=5000))])  # Bisagra L2 con penalización L2 (forma primal)



## Validación cruzada (GridSearchCV)

In [43]:
for n in ["Regresión logística", "SVM lineal"]:  # Modelos
    tabla, pliegues = MOD["cv"][n]  # Todas las combinaciones y desempeño por pliegue
    display(Markdown(f"**{n}**"))  # Título
    display(tabla.round(4))  # Malla completa (en el dashboard aparece resumida)
    display(pliegues.round(4))  # Mejor combinación por pliegue

**Regresión logística**

,C,class_weight,AUC-PR,AUC-PR desv.,AUC-ROC,AUC-ROC entrenamiento
2,0.0100,None,0.6433,0.0821,0.8985,0.9011
4,0.1000,None,0.6429,0.0819,0.8980,0.9009
6,1.0000,None,0.6422,0.0820,0.8973,0.9007
8,10.0000,None,0.6419,0.0821,0.8970,0.9007
0,0.0010,None,0.6414,0.0826,0.8998,0.9016
3,0.0100,balanced,0.6373,0.0828,0.9015,0.9040
5,0.1000,balanced,0.6370,0.0824,0.9010,0.9040
7,1.0000,balanced,0.6366,0.0825,0.9005,0.9040
9,10.0000,balanced,0.6365,0.0826,0.9004,0.9040
1,0.0010,balanced,0.6364,0.0840,0.9020,0.9036


,AUC-PR,AUC-ROC
2018,0.6089,0.8969
2019,0.7812,0.8956
2020,0.6184,0.8993
2021,0.5648,0.9022


**SVM lineal**

,C,class_weight,AUC-PR,AUC-PR desv.,AUC-ROC,AUC-ROC entrenamiento
4,0.0100,None,0.6438,0.0802,0.9000,0.9020
2,0.0010,None,0.6437,0.0802,0.9007,0.9024
6,0.1000,None,0.6435,0.0803,0.8995,0.9019
8,1.0000,None,0.6434,0.0803,0.8994,0.9019
0,0.0001,None,0.6406,0.0814,0.9029,0.9041
3,0.0010,balanced,0.6392,0.0828,0.9017,0.9037
5,0.0100,balanced,0.6391,0.0825,0.9012,0.9037
7,0.1000,balanced,0.6387,0.0826,0.9008,0.9037
9,1.0000,balanced,0.6386,0.0826,0.9007,0.9037
1,0.0001,balanced,0.6376,0.0841,0.9023,0.9035


,AUC-PR,AUC-ROC
2018,0.6127,0.8991
2019,0.7783,0.8950
2020,0.6177,0.9007
2021,0.5664,0.9052


**Interpretación.** Los dos modelos eligen $C = 0{,}01$ sin ponderar clases. El AUC-PR casi no cambia con $C$ (con 130 000 filas la regularización importa poco para el desempeño, pero estabiliza los coeficientes ante la colinealidad). Ponderar clases (`balanced`) baja el AUC-PR. El AUC-ROC de entrenamiento (≈ 0,901) y de validación (≈ 0,899) son casi iguales: no hay sobreajuste. El desempeño cambia más entre años (AUC-PR de 0,56 en 2021 a 0,78 en 2019) que entre modelos.

## Detalle: el escalado de Platt de la SVM

La SVM produce un margen $s = \mathbf w^\top \mathbf x + b$, no una probabilidad. Con la validación 2022 se ajusta una regresión logística de una variable, $\hat p = \sigma(a\,s + c)$ (Platt, 1999). Se hace con 2022 y no con el entrenamiento para no reutilizar los mismos datos que definieron el hiperplano.

In [44]:
a, c_ = MOD["platt"]  # Parámetros de Platt
print(f"p = σ({a:.3f} · margen + {c_:.3f})")  # Transformación
for s in (-2, -1, 0, 1):  # Ejemplos
    print(f"margen {s:+d} → probabilidad {1 / (1 + np.exp(-(a * s + c_))):.3f}")  # Probabilidad

p = σ(3.546 · margen + 0.263)
margen -2 → probabilidad 0.001
margen -1 → probabilidad 0.036
margen +0 → probabilidad 0.565
margen +1 → probabilidad 0.978


**Interpretación.** Un margen 0 (justo en el hiperplano) equivale a una probabilidad de ≈ 0,57; la pendiente alta ($a \approx 3{,}5$) indica que el margen ordena bien: pequeñas diferencias de margen cambian mucho la probabilidad.

## Resultados en la prueba 2023–2024

:::{admonition} Detalle: intervalos de confianza por bootstrap de conglomerados
En cada una de las 1 000 réplicas se eligen 376 municipios **con reemplazo**, se juntan todas sus semanas y se recalcula la métrica. El intervalo es el de los percentiles 2,5 y 97,5. Remuestrear municipios, no filas, respeta la dependencia entre las semanas de un mismo municipio y da intervalos honestos.
:::

In [45]:
tabla = pd.concat({n: r.apply(lambda f: f"{f['valor']:.3f} [{f['IC 2,5 %']:.3f}; {f['IC 97,5 %']:.3f}]", axis=1) for n, r in MOD["ic"].items()}, axis=1).T  # Valor [IC]
tabla

,AUC-ROC,AUC-PR,Accuracy,Precision,Recall,F1,MCC,Brier
Regresión logística,0.885 [0.875; 0.895],0.809 [0.782; 0.833],0.825 [0.814; 0.836],0.700 [0.677; 0.720],0.742 [0.718; 0.765],0.720 [0.698; 0.741],0.594 [0.568; 0.616],0.122 [0.115; 0.129]
SVM lineal,0.887 [0.876; 0.896],0.808 [0.781; 0.832],0.827 [0.815; 0.838],0.706 [0.683; 0.727],0.734 [0.710; 0.756],0.720 [0.697; 0.740],0.595 [0.569; 0.617],0.121 [0.114; 0.128]
Dummy (prior),0.500 [0.500; 0.500],0.303 [0.281; 0.327],0.697 [0.673; 0.719],0.000 [0.000; 0.000],0.000 [0.000; 0.000],0.000 [0.000; 0.000],0.000 [0.000; 0.000],0.251 [0.234; 0.270]
Persistencia,0.787 [0.774; 0.800],0.588 [0.557; 0.617],0.823 [0.812; 0.834],0.712 [0.687; 0.734],0.697 [0.673; 0.719],0.705 [0.680; 0.727],0.578 [0.551; 0.603],0.177 [0.166; 0.188]
Persistencia (8 sem.),0.859 [0.849; 0.869],0.728 [0.699; 0.754],0.805 [0.794; 0.816],0.655 [0.629; 0.678],0.756 [0.735; 0.775],0.701 [0.678; 0.723],0.561 [0.536; 0.584],0.135 [0.127; 0.142]
Ingenuo estacional,0.540 [0.527; 0.552],0.325 [0.300; 0.352],0.662 [0.642; 0.680],0.399 [0.359; 0.439],0.230 [0.203; 0.258],0.292 [0.260; 0.323],0.096 [0.065; 0.126],0.338 [0.320; 0.358]


**Interpretación.** Los dos modelos base superan a todas las líneas base en AUC-ROC (0,886–0,887 frente a 0,859 de la mejor), AUC-PR (0,81 frente a 0,73), F1 y MCC, con intervalos de unas ± 0,01. La exactitud no los distingue de la persistencia simple (≈ 0,82–0,83), y el Dummy ya llega a 0,70: por eso no se usa para decidir. El ingenuo estacional es casi inútil (AUC 0,54).

In [46]:
MOD["diferencias"].round(4)  # Diferencias pareadas

,comparación,métrica,diferencia,"IC 2,5 %","IC 97,5 %",p
0,Regresión logística − Persistencia (8 sem.),AUC-ROC,0.0262,0.0225,0.0296,0.0010
1,Regresión logística − Persistencia (8 sem.),AUC-PR,0.0813,0.0745,0.0883,0.0010
2,SVM lineal − Persistencia (8 sem.),AUC-ROC,0.0274,0.0244,0.0303,0.0010
3,SVM lineal − Persistencia (8 sem.),AUC-PR,0.0807,0.0746,0.0868,0.0010
4,Regresión logística − Persistencia,AUC-ROC,0.0980,0.0931,0.1036,0.0010
5,Regresión logística − Persistencia,AUC-PR,0.2206,0.2109,0.2290,0.0010
6,SVM lineal − Regresión logística,AUC-ROC,0.0012,0.0002,0.0022,0.0220
7,SVM lineal − Regresión logística,AUC-PR,-0.0006,-0.0020,0.0008,0.3560


**Interpretación.** Frente a la persistencia de 8 semanas, ambos modelos ganan ≈ 0,026–0,027 de AUC-ROC y ≈ 0,08 de AUC-PR, con intervalos que excluyen el 0. Entre ellos, la SVM gana 0,001 de AUC-ROC ($p \approx 0{,}02$): significativo por el tamaño de muestra, irrelevante en la práctica; en AUC-PR no hay diferencia. Son, en la práctica, el mismo modelo.

In [47]:
for n in ["Regresión logística", "SVM lineal"]:  # Matriz de confusión de cada modelo
    cm, met = metricas_umbral(n, MOD["umbrales"][n])  # Con su umbral de 2022
    mostrar(fg.fig_matriz(cm.tolist(), MOD["umbrales"][n]))  # Matriz
    print(n, {k: round(v, 3) for k, v in met.items()})  # Métricas

Regresión logística {'precisión': 0.7, 'recall': 0.742, 'F1': 0.72, 'exactitud': 0.825, 'MCC': 0.594, 'alertas por semana': 120.77}


SVM lineal {'precisión': 0.706, 'recall': 0.734, 'F1': 0.72, 'exactitud': 0.827, 'MCC': 0.595, 'alertas por semana': 118.49}


**Interpretación.** Con el umbral de máximo F1 en 2022, la regresión logística anticipa ≈ 74 % de los brotes con precisión ≈ 70 % (unas 121 alertas por semana en todo el país). La SVM tiene un umbral distinto (sus probabilidades vienen de Platt) pero un balance muy parecido. En el dashboard el umbral se mueve con un control deslizante.

In [48]:
mostrar(fg.fig_curvas(curvas(["Regresión logística", "SVM lineal", "Persistencia (8 sem.)", "Persistencia", "Ingenuo estacional", "Dummy (prior)"]), MOD["y_te"].mean()))

**Interpretación.** Las curvas de los dos modelos base se superponen y dominan a las líneas base en casi todo el rango. La persistencia simple es un solo punto en la ROC (es binaria).

## Diagnóstico

In [49]:
for n in ["Regresión logística", "SVM lineal"]:  # Calibración y brecha
    dg = MOD["diagnosticos"][n]  # Diagnóstico
    display(Markdown(f"**{n}**"))  # Título
    mostrar(fg.fig_calibracion(dg["calibracion"]))  # Calibración
    display(dg["brecha"].round(4))  # Mismas métricas en los tres conjuntos

**Regresión logística**

,AUC-ROC,AUC-PR,Accuracy,Precision,Recall,F1,MCC,Brier
Entrenamiento,0.9067,0.6532,0.9178,0.5950,0.6266,0.6104,0.5647,0.0558
Validación 2022,0.8978,0.6696,0.9115,0.6632,0.5751,0.6160,0.5681,0.0673
Prueba 2023–24,0.8855,0.8090,0.8254,0.6999,0.7418,0.7203,0.5940,0.1217


**SVM lineal**

,AUC-ROC,AUC-PR,Accuracy,Precision,Recall,F1,MCC,Brier
Entrenamiento,0.9079,0.6534,0.9194,0.6056,0.6186,0.6120,0.5671,0.0564
Validación 2022,0.9017,0.6741,0.9089,0.6391,0.6030,0.6205,0.5691,0.0657
Prueba 2023–24,0.8867,0.8084,0.8268,0.7060,0.7341,0.7198,0.5947,0.1209


**Interpretación.** En 2022 el error de calibración es pequeño (ECE ≈ 0,03); en 2023–2024 crece (ECE 0,04–0,05) y los modelos subestiman, porque la prevalencia triplica la del entrenamiento (*prior shift*). La brecha de AUC-ROC entre entrenamiento y prueba es de ≈ 0,02: no hay sobreajuste; las diferencias en AUC-PR, precisión y Brier se deben a la prevalencia.

:::{admonition} Detalle: error de calibración esperado
$\text{ECE} = \sum_{b=1}^{10} \frac{n_b}{n}\,\big|\bar y_b - \bar p_b\big|$ sobre deciles de probabilidad: el promedio, ponderado por tamaño, de la distancia entre lo predicho y lo observado.
:::

In [50]:
for n in ["Regresión logística", "SVM lineal"]:  # Residuos
    dg = MOD["diagnosticos"][n]  # Diagnóstico
    mostrar(fg.fig_acf_residuos(dg["acf"]))  # ACF de residuos
    print(n, "· ACF rezagos 1–4:", np.round(dg["acf"][:4], 3), "· Moran del residuo medio:", round(dg["moran"]["I"], 3), "p =", dg["moran"]["p"])  # Resumen
mostrar(fg.fig_residuo_mapa(M, MOD["diagnosticos"]["Regresión logística"]["residuo_mun"]))  # Mapa de residuos

Regresión logística · ACF rezagos 1–4: [0.339 0.261 0.19  0.01 ] · Moran del residuo medio: 0.158 p = 0.001


SVM lineal · ACF rezagos 1–4: [ 0.34   0.261  0.185 -0.029] · Moran del residuo medio: 0.174 p = 0.001


**Interpretación.** Los residuos tienen autocorrelación temporal en los rezagos 1–3 (≈ 0,34, 0,26 y 0,19; desaparece en el rezago 4) y espacial (I de Moran ≈ 0,16–0,17, $p = 0{,}001$). Hay información estructural sin capturar: los episodios de brote duran varias semanas y se propagan entre vecinos. Es la motivación para agregar rezagos espaciales y modelos no lineales.

In [51]:
mostrar(fg.fig_curva_aprendizaje(MOD["curva"], "AUC-ROC"))  # Curva de aprendizaje
MOD["curva"].round(4)

,fraccion,filas,AUC-PR entrenamiento,AUC-PR validación,AUC-ROC entrenamiento,AUC-ROC validación
0,0.1000,13414,0.7160,0.6722,0.9202,0.9028
1,0.2000,26475,0.6758,0.6730,0.9163,0.8977
2,0.3500,46596,0.6513,0.6721,0.9088,0.8982
3,0.5000,66364,0.6356,0.6708,0.9075,0.8998
4,0.7500,99546,0.6417,0.6714,0.9070,0.9005
5,1.0000,132728,0.6532,0.6696,0.9067,0.8978


**Interpretación.** Con el 10 % de los municipios (13 000 filas) el AUC de validación ya es ≈ 0,90: la curva es plana y las curvas de entrenamiento y validación convergen. Más datos no mejorarían un modelo lineal; el límite es de sesgo.

In [52]:
MOD["bloques"].round(4)  # Validación por bloques de departamentos (zona de separación de 50 km)

,departamentos,municipios evaluados,excluidos por 50 km,LR zonas nuevas,LR zonas nuevas + futuro,SVM zonas nuevas,SVM zonas nuevas + futuro
bloque,,,,,,,
1,7,76,32,0.9056,0.8826,0.9101,0.8845
2,6,75,44,0.9008,0.8639,0.9011,0.8662
3,6,75,39,0.8987,0.9044,0.8999,0.9052
4,7,75,52,0.9276,0.8952,0.9277,0.8931
5,6,75,27,0.8834,0.8695,0.8876,0.8751


**Interpretación.** En departamentos no vistos el AUC-ROC es 0,88–0,93 (mismo periodo) y 0,86–0,91 (zonas nuevas y futuro): los modelos generalizan a zonas nuevas porque sus predictoras son relativas a cada municipio. Los dos modelos se comportan igual en cada bloque.

## Interpretación de los coeficientes

In [53]:
mostrar(fg.fig_coeficientes(MOD["coef_lr"]))  # Regresión logística
coef = MOD["coef_lr"].assign(odds_ratio=lambda t: np.exp(t["coef"]))  # Razón de odds
coef.reindex(coef["coef"].abs().sort_values(ascending=False).index).round(3)  # Tabla completa

,variable,coef,"IC 2,5 %","IC 97,5 %",odds_ratio
8,brote_m8,0.6020,0.4810,0.6580,1.8260
1,lc_l1,0.4340,0.3720,0.5000,1.5430
9,ratio_umbral,0.4290,0.3810,0.4850,1.5360
6,lc_m12,-0.4020,-0.5330,-0.2310,0.6690
4,lc_m4,0.2580,0.2300,0.2850,1.2940
0,lc_l0,0.2440,0.1810,0.2970,1.2770
27,oni_anom,0.2300,0.1710,0.2870,1.2590
23,ndvi_mean_m12,-0.2040,-0.3070,-0.1160,0.8150
2,lc_l2,0.1970,0.1510,0.2520,1.2180
5,lc_m8,-0.1970,-0.3190,-0.0500,0.8210


**Interpretación.** La proporción reciente de semanas en brote (odds × 1,83 por desviación estándar), el log de casos de la semana anterior (× 1,54) y los casos frente al umbral objetivo (× 1,54) son los factores de riesgo más fuertes. Las medias largas de casos tienen signo negativo condicional a las cortas: lo que anticipa un brote es la **aceleración** reciente. El ONI multiplica las odds por ≈ 1,26. Los coeficientes de temperatura incluyen el 0 (el 64 % está imputada) y la lluvia tiene signos mezclados entre ventanas: efectos no lineales que un modelo lineal no capta.

In [54]:
mostrar(fg.fig_pesos_svm(MOD["coef_svm"]))  # SVM lineal
lr = MOD["coef_lr"].set_index("variable")["coef"]  # Coeficientes logísticos
print(f"Correlación entre los pesos de la SVM y los coeficientes logísticos: {np.corrcoef(MOD['coef_svm'][lr.index], lr)[0, 1]:.2f}")  # Similitud
pd.DataFrame({"SVM": MOD["coef_svm"], "logística": lr}).sort_values("SVM", key=np.abs, ascending=False).head(10).round(3)

Correlación entre los pesos de la SVM y los coeficientes logísticos: 0.87


,SVM,logística
brote_m8,0.2100,0.6020
temp_mean_m8,0.1480,0.0900
lc_m12,-0.1300,-0.4020
temp_mean_m12,-0.1130,-0.0700
lc_l1,0.1080,0.4340
ratio_umbral,0.0710,0.4290
lc_m8,-0.0660,-0.1970
lc_m4,0.0630,0.2580
brote_l0,0.0630,0.0870
lc_l2,0.0550,0.1970


**Interpretación.** Los pesos de la SVM se parecen a los coeficientes logísticos (correlación ≈ 0,87) y la variable más pesada es la misma. La diferencia más visible está en la temperatura: la SVM reparte su efecto entre las ventanas de 8 y 12 semanas con signos opuestos, un síntoma de la colinealidad entre medias móviles. Las escalas no son comparables (la bisagra no mide log-odds).

## Simulador

El simulador del dashboard toma la semana con más casos de cada municipio en 2023–2024, cambia la temperatura (suma $\Delta$ a todas sus ventanas y a su anomalía), la lluvia (la multiplica) y el ONI, y recalcula $\hat p = \sigma(\boldsymbol\beta^\top \mathbf z + \beta_0)$ con la estandarización y los coeficientes del Pipeline logístico.

In [55]:
from tablero.datos import simular  # Misma función del dashboard
filas = MOD["simulador"]["filas"]  # Escenarios
probs = np.array([simular(f) for _, f in filas.iterrows()])  # Probabilidad observada de cada escenario
cali = filas.iloc[int(np.argmin(np.abs(probs - MOD["simulador"]["umbral"])))]  # Municipio más cercano al umbral de alerta
print(f"Escenario: {cali['municipio']} ({cali['departamento']}), semana del {cali['week_start'].date()} · Cali en su pico: {simular(filas[filas['COD_MUN_N'] == 76001].iloc[0]):.3f}")
pd.DataFrame({esc: {"probabilidad": simular(cali, *args)} for esc, args in {
    "clima observado": (0, 1, None), "+2 °C": (2, 1, None), "lluvia × 1,5": (0, 1.5, None), "El Niño fuerte (ONI 2)": (0, 1, 2.0),
    "La Niña fuerte (ONI −1,5)": (0, 1, -1.5)}.items()}).T.round(3)

Escenario: Cáqueza (Cundinamarca), semana del 2024-02-12 · Cali en su pico: 0.995


,probabilidad
clima observado,0.2410
+2 °C,0.2370
"lluvia × 1,5",0.2380
El Niño fuerte (ONI 2),0.2580
"La Niña fuerte (ONI −1,5)",0.1280


**Interpretación.** En una semana epidémica como el pico de Cali la probabilidad está saturada (≈ 0,995) y el clima no la mueve. Cerca del umbral sí importa: en Cáqueza (probabilidad 0,241, justo debajo del umbral 0,244) un El Niño fuerte la sube a 0,258 y activa la alerta, y La Niña la reduce casi a la mitad. En cambio, +2 °C o 50 % más lluvia apenas la mueven (la temperatura incluso la baja un poco), reflejo de coeficientes climáticos pequeños y de signos mezclados. El simulador muestra una asociación aprendida, no un efecto causal.

## Nota crítica

La exactitud en la prueba (≈ 0,83) está en el rango 80–90 % que la guía pide revisar: (1) la auditoría no encontró fuga; (2) el Dummy acierta 0,70 y la persistencia 0,82, así que la exactitud no mide el mérito del modelo; (3) la validación respeta el tiempo y el espacio; y (4) el problema no es trivial: la mejor regla simple llega a AUC-PR 0,73 y los modelos a 0,81, con residuos que aún tienen estructura. El desempeño alto en exactitud se explica por el desbalance y la persistencia natural de las epidemias.

# El dashboard

## Estructura

El dashboard (Dash 2.18 + Plotly 5.24) tiene tres pestañas. Cada resultado va acompañado de un recuadro de **interpretación**.

| Pestaña | Sub-pestañas | Contenido |
|---|---|---|
| **1. Contexto del problema** | — | Importancia, pregunta y objetivos, tipo de tarea, formulación matemática del objetivo y de los dos modelos base, fuente y licencia, diccionario de variables, partición con purga, casos por año y ética |
| **2. EDA** | Datos y calidad · Variable objetivo · Univariado · Bivariado · Multivariado · Fuga de datos · Temporal · Espacial · Espacio-temporal · Preprocesamiento | Las secciones 1 y 2.1–2.9 de la guía del primer entregable, con controles interactivos |
| **3. Modelos base** | Entrenamiento · Resultados · Diagnóstico · Interpretación · Simulador · Nota crítica | Regresión logística y SVM lineal frente a cuatro líneas base triviales |

## Componentes interactivos

| Dónde | Control | Qué hace |
|---|---|---|
| EDA › Variable objetivo | Municipio | Dibuja el canal endémico de cualquiera de los 376 municipios |
| EDA › Univariado | Variable | Histograma y caja de cualquier variable original o predictora |
| EDA › Bivariado | Par de variables, métrica, variable | Densidad conjunta, asociación con el objetivo (información mutua, biserial de rangos, AUC) y tasa de brote por decil |
| EDA › Multivariado | Color | t-SNE coloreado por objetivo o por región |
| EDA › Temporal | Agregación | Serie nacional por semana, mes o año |
| EDA › Espacial | Variable, indicador | Mapa de incidencia, brote, temperatura, lluvia o flujo; LISA o Getis-Ord |
| EDA › Espacio-temporal | Año | Mapa de la incidencia de cada año 2015–2021 |
| Modelos base › Resultados | Modelo y umbral | Matriz de confusión, precisión, recall, F1, MCC y alertas por semana con cualquier umbral |
| Modelos base › Resultados | Modelos | Curvas ROC y PR de los modelos elegidos |
| Modelos base › Diagnóstico | Modelo, métrica | Calibración, sobreajuste, residuos y curva de aprendizaje |
| Modelos base › Simulador | Municipio, temperatura, lluvia, ONI | Recalcula la probabilidad de brote con el modelo logístico entrenado |

## Arquitectura

```text
entregable2/
├── precalculo.py          # Calcula todo una vez y guarda tablero/artefactos.joblib (≈ 10 min)
├── tablero/               # La aplicación Dash
│   ├── app.py             # Encabezado, pestañas y pie de página
│   ├── datos.py           # Carga los artefactos; matriz de confusión, curvas y simulador en vivo
│   ├── figuras.py         # Todas las figuras (las usa también este libro)
│   ├── componentes.py     # Tarjetas, indicadores, tablas, fórmulas e interpretaciones
│   ├── sec_contexto.py    # Pestaña 1
│   ├── sec_eda.py         # Pestaña 2 y sus callbacks
│   ├── sec_modelos.py     # Pestaña 3 y sus callbacks
│   └── assets/            # Estilos (tema oscuro) e imágenes de fondo
├── construir/             # Generadores de los cuadernos de este libro
└── cuadernos/             # Capítulos ejecutados
```

**Decisión de diseño.** Todo lo costoso (EDA, GridSearchCV, bootstrap de 1 000 réplicas, pruebas espaciales) se calcula una vez en
`precalculo.py`. El dashboard solo lee el archivo de artefactos: abre en segundos, no reentrena nada y consume poca memoria, lo que lo
hace apto para un plan gratuito de despliegue. Los únicos cálculos en vivo son baratos: la matriz de confusión con el umbral elegido, las
curvas ROC/PR y el simulador (un producto punto con los coeficientes del Pipeline).

**Una sola fuente de verdad.** Este libro importa `tablero.figuras` y `tablero.datos`: las figuras y cifras del informe son las mismas
del dashboard.

## Ejecución

Desde la carpeta `entregable2` (con el entorno del proyecto):

```bash
python precalculo.py
```

```bash
python -m tablero.app
```

y abrir `http://localhost:8050`.


# Conclusiones y reproducibilidad

## Conclusiones

1. **Los dos modelos base lineales son equivalentes.** La regresión logística (AUC-ROC 0,885, AUC-PR 0,809) y la SVM lineal (0,887 y
   0,808) llegan casi al mismo hiperplano con las mismas variables: la diferencia en AUC-ROC (+0,001) es estadísticamente detectable pero
   irrelevante en la práctica, y en AUC-PR no hay diferencia.
2. **Superan con claridad a las líneas base triviales.** Frente a la persistencia de 8 semanas ganan ≈ 0,027 de AUC-ROC y ≈ 0,08 de
   AUC-PR (intervalos por municipio que excluyen el 0); frente a la persistencia simple, ≈ 0,10 y ≈ 0,22.
3. **La exactitud no sirve para juzgarlos.** Con ≈ 30 % de brotes en la prueba, el Dummy acierta 70 % y la persistencia 82 %.
4. **La historia de casos domina** y lo que anticipa un brote es su aceleración reciente; El Niño aporta; la temperatura local casi no
   aporta en un modelo lineal (64 % imputada y relación no lineal).
5. **No hay sobreajuste; el límite es de sesgo.** Curva de aprendizaje plana, brecha pequeña entre entrenamiento y prueba, pero residuos
   con autocorrelación temporal y espacial: falta la dinámica de vecindad (Knox, Moran) y la no linealidad del clima.
6. **Generalizan a zonas nuevas** (bloques de departamentos con 50 km de separación) porque sus variables son relativas a cada municipio.
7. **Las probabilidades se desplazan con la prevalencia.** En 2023–2024 quedan subestimadas: el umbral o la intersección deben
   recalibrarse con datos recientes.

## Limitaciones

* Representatividad: 376 de 1 122 municipios (≈ 60 % de la población), con el Caribe y la Orinoquía sobrerrepresentados.
* Subregistro variable de la notificación y caída de 2020 por la pandemia.
* Temperatura imputada en los municipios pequeños; coordenadas como centroides, sin polígonos.
* Dos años de prueba, uno de ellos (2024) excepcional.

## Siguientes pasos

* Variables de rezago espacial (vecinos en $t$) y de interacción con El Niño.
* Modelos no lineales y ensambles; recalibración periódica del umbral.

## Reproducibilidad

* **Semilla:** `SEMILLA = 42` (`src/config.py`).
* **Entorno:** Python 3.9.25 con las versiones de `entregable2/requirements.txt`.
* **Pasos** (desde `entregable2`):

```bash
python precalculo.py
```

```bash
python -m tablero.app
```

```bash
cd construir && python construir_todo.py && cd ../cuadernos
```

```bash
for n in *.ipynb; do jupyter nbconvert --to notebook --execute --inplace $n; done
```

```bash
cd ../.. && jupyter-book build entregable2
```
